In [75]:
import sys
import subprocess
import os
import cantools
from pathlib import Path
import inspect

OUTPUT = "../aim/nemesis.dbc"
DBC_DIR = "../dbc"

In [ ]:
ECU_SIGNALS = {
    "abs_active":                           None,
    "abs_armed":                            None,
    "accelerator_pedal_position":           "apps",
    "air_temperature":                      None,
    "barometric_pressure":                  "barometric_pres",
    "battery_voltage":                      None,
    "brake_pressure_difference":            "brake_pres_diff",
    "brake_pressure_front":                 "brake_pres_front",
    "brake_pressure_rear":                  "brake_pres_rear",
    "clutch_switch":                        None,
    "coolant_temperature":                  "coolant_temp",
    "cut_percentage":                       None,
    "driveshaft_rpm":                       None,
    "ecu_temperature":                      None,
    "engine_demand":                        None,
    "engine_limiting_active":               "engine_limit_act",
    "engine_protection_reason":             "engine_prot_reas",
    "fuel_flow":                            None,
    "fuel_level":                           None,
    "fuel_pressure":                        None,
    "fuel_temperature":                     None,
    "total_fuel_used":                      None,
    "gear":                                 None,
    "launch_control_active":                "launch_cont_act",
    "launch_control_switch":                "launch_cont_sw",
    "manifold_pressure":                    "map",
    "oil_pressure":                         None,
    "oil_temperature":                      None,
    "rpm":                                  "engine_rpm",
    "steering_wheel_angle":                 "steering_angle",
    "target_lambda":                        None,
    "throttle_position":                    "tps",
    "traction_control_active":              "traction_con_act",
    "traction_control_enabled":             "traction_con_en",
    "vehicle_speed":                        None,
    "wheel_diff":                           None,
    "wheel_slip":                           None,
    "wheel_speed_front_left":               "wheel_speed_fl",
    "wheel_speed_front_right":              "wheel_speed_fr",
    "wheel_speed_rear_left":                "wheel_speed_rl",
    "wheel_speed_rear_right":               "wheel_speed_rr",
    "wideband_overall":                     None,
    "shock_travel_sensor_front_left":       "damper_travel_fl",
    "shock_travel_sensor_front_right":      "damper_travel_fr",
    "shock_travel_sensor_rear_left":        "damper_travel_rl",
    "shock_travel_sensor_rear_right":       "damper_travel_rr",
    "lateral_g":                            "ecu_lat_acc",
    "longitudinal_g":                       "ecu_lon_acc",
    "vertical_g":                           "ecu_ver_acc",
}

AERO_SIGNALS = {  
    "Aero_Static_1_P1":                     "Aero_Static_P1",
    "Aero_Static_1_P2":                     "Aero_Static_P2",
    "Aero_Static_1_P3":                     "Aero_Static_P3",
    "Aero_Static_1_P4":                     "Aero_Static_P4",
    "Aero_Static_1_P5":                     "Aero_Static_P5",
    "Aero_Static_1_P6":                     "Aero_Static_P6",
    "Aero_Static_1_P7":                     "Aero_Static_P7",
    "Aero_Static_1_P8":                     "Aero_Static_P8",
    "Aero_Static_1_P_REF":                  "Aero_Static_REF",
    "P_Scanner_1_Temperature":              "P_Scanner_Temp",
}

IMU_SIGNALS = {
    "front_acc_x":                          None,
    "front_acc_y":                          None,
    "front_acc_y":                          None,
    "front_angular_x":                      None,
    "front_angular_y":                      None,
    "front_angular_z":                      None,
    "front_imu_temp":                       None,
    "rear_acc_x":                           None,
    "rear_acc_y":                           None,
    "rear_acc_y":                           None,
    "rear_angular_x":                       None,
    "rear_angular_y":                       None,
    "rear_angular_z":                       None,
    "rear_imu_temp":                        None,
}

TCS_SIGNALS = {
    "auto_shift":                           None,
    "auto_shift_rpm_1":                     None,
    "auto_shift_rpm_2":                     None,    
    "auto_shift_rpm_3":                     None,
    "auto_shift_rpm_4":                     None,
    "servo_percent":                        None,
    "servo_position":                       None,
    "servo_potentiometer":                  "servo_pot",
}

SWC_SIGNALS = {
    "clutch_left":                          None,
    "clutch_right":                         None,
    "clutch_left_raw":                      None,
    "clutch_right_raw":                     None,
}

ACC_SIGNALS = {
    "acc_1":                                None,
    "acc_2":                                None,
    "acc_3":                                None,
    "acc_4":                                None,
    "acc_5":                                None,
    "acc_6":                                None,
    "acc_7":                                None,
    "acc_8":                                None,
    "acc_9":                                None,
    "acc_10":                               None,
    "acc_11":                               None,
    "acc_12":                               None,
    "acc_13":                               None,
    "acc_14":                               None,
    "acc_15":                               None,
    "acc_16":                               None,
}

DISPLAY_SIGNALS = {
    "cpu_temperature":                      "display_temp",
}

In [77]:
no_rename_over = [(orig, len(orig)) for orig, rename in SIGNALS.items() if rename is None and len(orig) > 16]
renamed_over   = [(orig, rename, len(rename)) for orig, rename in SIGNALS.items() if rename is not None and len(rename) > 16]

if no_rename_over:
    print("No rename, original exceeds 16 chars:")
    for orig, n in no_rename_over:
        print(f"  {orig!r} ({n} chars)")

if renamed_over:
    print("Renamed, but rename still exceeds 16 chars:")
    for orig, rename, n in renamed_over:
        print(f"  {orig!r} -> {rename!r} ({n} chars)")

In [78]:
if os.path.exists(OUTPUT):
    os.remove(OUTPUT)

SIGNAL_RENAME = {
    orig.lower(): (rename if rename is not None else orig)
    for orig, rename in SIGNALS.items()
}

def clone_with_kwargs(cls, candidates):
    valid = set(inspect.signature(cls.__init__).parameters) - {"self"}
    return cls(**{k: v for k, v in candidates.items() if k in valid})

def clone_signal(signal, new_name):
    return clone_with_kwargs(cantools.database.Signal, {
        "name": new_name,
        "start": signal.start,
        "length": signal.length,
        "byte_order": signal.byte_order,
        "is_signed": signal.is_signed,
        "scale": signal.scale,
        "offset": signal.offset,
        "minimum": signal.minimum,
        "maximum": signal.maximum,
        "unit": signal.unit,
        "choices": signal.choices,
        "comment": signal.comment,
        "nodes": getattr(signal, "nodes", None),
        "is_multiplexer": signal.is_multiplexer,
        "multiplexer_ids": signal.multiplexer_ids,
        "multiplexer_signal": getattr(signal, "multiplexer_signal", None),
        "is_float": getattr(signal, "is_float", False),
        "decimal": getattr(signal, "decimal", None),
        "dbc_specifics": getattr(signal, "dbc_specifics", None),
        "spn": getattr(signal, "spn", None),
    })


def clone_message(message, signals):
    return clone_with_kwargs(cantools.database.Message, {
        "frame_id": message.frame_id,
        "name": message.name,
        "length": message.length,
        "signals": signals,
        "comment": message.comment,
        "senders": message.senders,
        "send_type": message.send_type,
        "cycle_time": message.cycle_time,
        "bus_name": message.bus_name,
        "signal_groups": getattr(message, "signal_groups", None),
        "is_extended_frame": getattr(message, "is_extended_frame", False),
        "is_fd": getattr(message, "is_fd", False),
        "is_container": getattr(message, "is_container", False),
        "header_id": getattr(message, "header_id", None),
        "unused_bit_pattern": getattr(message, "unused_bit_pattern", 0),
        "strict": getattr(message, "strict", False),
        "sort_signals": None,
        "dbc_specifics": getattr(message, "dbc_specifics", None),
    })

def filter_signals(signals):
    return [
        clone_signal(s, SIGNAL_RENAME[s.name.lower()])
        for s in signals
        if SIGNAL_RENAME.get(s.name.lower()) is not None
    ]

def merge_message(existing, incoming_signals):
    existing_names = {s.name for s in existing.signals}
    new = [s for s in incoming_signals if s.name not in existing_names]
    return clone_message(existing, existing.signals + new) if new else existing

dbc_files = sorted(Path(DBC_DIR).glob("*.dbc"))
if not dbc_files:
    raise FileNotFoundError(f"No .dbc files found in {DBC_DIR}")

all_messages = {}

for dbc_file in dbc_files:
    db = cantools.database.load_file(str(dbc_file))
    for msg in db.messages:
        all_messages[msg.frame_id] = (
            msg if msg.frame_id not in all_messages
            else merge_message(all_messages[msg.frame_id], msg.signals)
        )

base_messages = {
    m.frame_id: clone_message(m, filtered)
    for m in all_messages.values()
    if (filtered := filter_signals(m.signals))
}

output_db = cantools.database.Database(messages=list(base_messages.values()))
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
cantools.database.dump_file(output_db, OUTPUT)

print(f"Done. Saved to {OUTPUT}")

Done. Saved to ../aim/nemesis.dbc
